In [1]:
# Comprehensive marker testing for poolparty
import poolparty as pp
from poolparty.marker_ops import (
    find_all_markers,
    has_marker,
    validate_single_marker,
    strip_all_markers,
    get_length_without_markers,
    get_nonmarker_positions,
    build_marker_tag,
)

In [2]:
# --- Parsing: find_all_markers with various marker types ---
# Zero-length marker
m1 = find_all_markers('ACGT<ins/>TT')
assert len(m1) == 1 and m1[0].name == 'ins' and m1[0].content == '' and m1[0].is_zero_length

# Region marker with content
m2 = find_all_markers('AC<region>TG</region>AA')
assert len(m2) == 1 and m2[0].name == 'region' and m2[0].content == 'TG' and not m2[0].is_zero_length

# Strand attribute
m3 = find_all_markers("AC<region strand='-'>TG</region>AA")
assert m3[0].strand == '-'

# seq_length attribute (integer)
m4 = find_all_markers("<orf seq_length='4'>ACGT</orf>")
assert m4[0].declared_seq_length_str == '4'

# seq_length='None' for variable length
m5 = find_all_markers("<var seq_length='None'>ACGT</var>")
assert m5[0].declared_seq_length_str == 'None' and m5[0].is_variable_length

# Multiple markers
m6 = find_all_markers('A<m1>B</m1>C<m2/>D')
assert len(m6) == 2 and m6[0].name == 'm1' and m6[1].name == 'm2'

# has_marker and validate_single_marker
assert has_marker('AC<region>TG</region>AA', 'region') == True
assert has_marker('ACGT', 'region') == False
vm = validate_single_marker('AC<region>TG</region>AA', 'region')
assert vm.name == 'region' and vm.content == 'TG'

In [3]:
# --- Parsing: string manipulation utilities ---
# strip_all_markers removes tags but keeps content
assert strip_all_markers('AC<region>.TG.</region>AA') == 'AC.TG.AA'
assert strip_all_markers('AC<ins/>TG') == 'ACTG'
assert strip_all_markers('A<m1>B</m1>C<m2/>D') == 'ABCD'

# get_length_without_markers returns sequence length excluding tags
assert get_length_without_markers('ACGT') == 4
#assert get_length_without_markers('A.CG.T') == 4 THIS FAILS! TODO: Have this return only valid seq length
assert get_length_without_markers('AC<region>TG</region>AA') == 6
assert get_length_without_markers('AC<ins/>GT') == 4

# get_nonmarker_positions returns raw string positions of non-tag characters
assert get_nonmarker_positions('ACGT') == [0, 1, 2, 3]
pos = get_nonmarker_positions('AC<ins/>GT')  # A=0, C=1, <ins/>=2-7, G=8, T=9
assert pos == [0, 1, 8, 9]

# build_marker_tag constructs marker strings
assert build_marker_tag('ins', '') == '<ins/>'
assert build_marker_tag('ins', '', '-') == "<ins strand='-'/>"
assert build_marker_tag('region', 'ACGT') == '<region>ACGT</region>'
assert build_marker_tag('region', 'ACGT', '-') == "<region strand='-'>ACGT</region>"
assert build_marker_tag('orf', 'ACGT', '+', seq_length=4) == "<orf seq_length='4'>ACGT</orf>"
assert build_marker_tag('var', 'ACGT', '+', seq_length=-1) == "<var seq_length='None'>ACGT</var>"

In [4]:
# --- insert_marker: fixed position markers ---
pp.init()

# Zero-length marker at position 4
p1 = pp.insert_marker('ACGTACGT', 'ins', start=4)
df1 = p1.generate_library(num_cycles=1)
print(f"{df1['seq'].iloc[0]=}")
assert df1['seq'].iloc[0] == 'ACGT<ins/>ACGT'

# Region marker encompassing positions 2-5
pp.init()
p2 = pp.insert_marker('ACGTACGT', 'region', start=2, stop=5)
df2 = p2.generate_library(num_cycles=1)
print(f"{df2['seq'].iloc[0]=}")
assert df2['seq'].iloc[0] == 'AC<region>GTA</region>CGT'

# Marker with strand='-'
pp.init()
p3 = pp.insert_marker('ACGT', 'site', start=1, stop=3, strand='-')
df3 = p3.generate_library(num_cycles=1)
print(f"{df3['seq'].iloc[0]=}")
assert df3['seq'].iloc[0] == "A<site strand='-'>CG</site>T"

df1['seq'].iloc[0]='ACGT<ins/>ACGT'
df2['seq'].iloc[0]='AC<region>GTA</region>CGT'
df3['seq'].iloc[0]="A<site strand='-'>CG</site>T"


In [5]:
# --- marker_scan: sequential mode ---
pp.init()

# Zero-length markers at all positions (5 positions for 4-char seq)
p1 = pp.marker_scan('ACGT', marker='m', mode='sequential')
df1 = p1.generate_library(num_cycles=1)
assert len(df1) == 5
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
assert '<m/>ACGT' in seqs1 and 'A<m/>CGT' in seqs1 and 'ACGT<m/>' in seqs1

# Region markers with marker_length=2 (3 positions)
pp.init()
p2 = pp.marker_scan('ACGT', marker='r', mode='sequential', marker_length=2)
df2 = p2.generate_library(num_cycles=1)
assert len(df2) == 3
seqs2 = list(df2['seq'])
print(f'{seqs2=}')
assert '<r>AC</r>GT' in seqs2 and 'A<r>CG</r>T' in seqs2 and 'AC<r>GT</r>' in seqs2

# Position filtering - only positions 1 and 3
pp.init()
p3 = pp.marker_scan('ACGT', marker='m', mode='sequential', positions=[1, 3])
df3 = p3.generate_library(num_cycles=1)
assert len(df3) == 2
seqs3 = list(df3['seq'])
print(f'{seqs3=}')
assert 'A<m/>CGT' in seqs3 and 'ACG<m/>T' in seqs3

seqs1=['<m/>ACGT', 'A<m/>CGT', 'AC<m/>GT', 'ACG<m/>T', 'ACGT<m/>']
seqs2=['<r>AC</r>GT', 'A<r>CG</r>T', 'AC<r>GT</r>']
seqs3=['A<m/>CGT', 'ACG<m/>T']


In [6]:
# --- marker_scan: random mode and strand options ---
pp.init()

# Random mode - each sequence has marker at random position
p1 = pp.marker_scan('ACGTACGT', marker='m', mode='random')
df1 = p1.generate_library(num_seqs=10, seed=42)
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
for seq in df1['seq']:
    assert '<m/>' in seq

# strand='both' doubles states (5 positions x 2 strands = 10)
pp.init()
p2 = pp.marker_scan('ACGT', marker='m', mode='sequential', strand='both')
df2 = p2.generate_library(num_cycles=1)
assert len(df2) == 10
# Check both strands appear
plus_count = sum("strand='+'" in s for s in df2['seq'])
minus_count = sum("strand='-'" in s for s in df2['seq'])
assert plus_count == 5 and minus_count == 5
seqs2 = list(df2['seq'])
print(f'{seqs2=}')

# strand='-' only
pp.init()
p3 = pp.marker_scan('ACGT', marker='m', mode='sequential', strand='-')
df3 = p3.generate_library(num_cycles=1)
assert len(df3) == 5
for seq in df3['seq']:
    assert "strand='-'" in seq
seqs3 = list(df3['seq'])
print(f'{seqs3=}')

seqs1=['<m/>ACGTACGT', 'ACGTAC<m/>GT', 'ACGTA<m/>CGT', 'ACG<m/>TACGT', 'ACG<m/>TACGT', 'ACGTACG<m/>T', '<m/>ACGTACGT', 'ACGTAC<m/>GT', 'A<m/>CGTACGT', '<m/>ACGTACGT']
seqs2=["<m strand='+'/>ACGT", "<m strand='-'/>ACGT", "A<m strand='+'/>CGT", "A<m strand='-'/>CGT", "AC<m strand='+'/>GT", "AC<m strand='-'/>GT", "ACG<m strand='+'/>T", "ACG<m strand='-'/>T", "ACGT<m strand='+'/>", "ACGT<m strand='-'/>"]
seqs3=["<m strand='-'/>ACGT", "A<m strand='-'/>CGT", "AC<m strand='-'/>GT", "ACG<m strand='-'/>T", "ACGT<m strand='-'/>"]


In [7]:
# --- marker_multiscan: multiple marker insertion ---
pp.init()

# Insert 2 markers with ordered mode (markers[i] -> position[i])
p1 = pp.marker_multiscan('ACGTACGTACGT', markers=['m1', 'm2'], num_insertions=2, mode='random')
df1 = p1.generate_library(num_seqs=5, seed=42)
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
for seq in df1['seq']:
    assert '<m1/>' in seq and '<m2/>' in seq

# With marker_length > 0 for region markers
pp.init()
p2 = pp.marker_multiscan('ACGTACGTACGT', markers=['a', 'b'], num_insertions=2, 
                          marker_length=2, mode='random')
df2 = p2.generate_library(num_seqs=5, seed=42)
seqs2 = list(df2['seq'])
print(f'{seqs2=}')
for seq in df2['seq']:
    # Both region markers should be present with 2-char content
    try:
        assert '<a>' in seq and '</a>' in seq
        assert '<b>' in seq and '</b>' in seq
    except:
        print(f'error: {seq=}')


seqs1=['A<m1/>CGTACGTAC<m2/>GT', 'ACGTA<m1/>CGTACGT<m2/>', 'A<m1/>CGTACGTA<m2/>CGT', 'A<m1/>CGTAC<m2/>GTACGT', 'ACGTACGT<m1/>A<m2/>CGT']
seqs2=['<a>AC</a>GTAC<b>GT</b>ACGT', 'ACGT<a>AC</a>GTA<b>CG</b>T', '<a>AC</a>GTACGTAC<b>GT</b>', 'AC<a>GT</a>ACGTA<b>CG</b>T', 'ACG<a>TA</a>C<b>GT</b>ACGT']


In [8]:
# --- extract_marker_content ---
pp.init()

# Basic extraction
bg1 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
content1 = pp.extract_marker_content(bg1, 'region')
df1 = content1.generate_library(num_cycles=1)
assert df1['seq'].iloc[0] == 'TTAA'
seqs1 = list(df1['seq'])
print(f'{seqs1=}')

# Extraction with strand='-' reverse complements the content
pp.init()
bg2 = pp.from_seq("ACGT<region strand='-'>AACG</region>GCGC")
content2 = pp.extract_marker_content(bg2, 'region')
df2 = content2.generate_library(num_cycles=1)
# AACG reverse complement = CGTT
assert df2['seq'].iloc[0] == 'CGTT'
seqs2 = list(df2['seq'])
print(f'{seqs2=}')

# Extracted content pool has correct seq_length
pp.init()
bg3 = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
content3 = pp.extract_marker_content(bg3, 'orf')
df3 = content3.generate_library(num_cycles=1)
assert content3.seq_length == 6
seqs3 = list(df3['seq'])
print(f'{seqs3=}')

seqs1=['TTAA']
seqs2=['CGTT']
seqs3=['ATGCCC']


In [9]:
# --- replace_marker_content ---
pp.init()

# Replace zero-length marker with content
bg1 = pp.from_seq('ACGT<insert/>TTTT')
inserts1 = pp.from_seqs(['AAA', 'GGG'], mode='sequential')
result1 = pp.replace_marker_content(bg1, inserts1, 'insert')
df1 = result1.generate_library(num_cycles=1)
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
assert 'ACGTAAATTTT' in seqs1 and 'ACGTGGGTTTT' in seqs1

# Replace region marker (replaces existing content)
pp.init()
bg2 = pp.from_seq('PREFIX<var>OLDCONTENT</var>SUFFIX')
variants = pp.from_seqs(['NEW1', 'NEW2'], mode='sequential')
result2 = pp.replace_marker_content(bg2, variants, 'var')
df2 = result2.generate_library(num_cycles=1)
seqs2 = list(df2['seq'])
print(f'{seqs2=}')
assert 'PREFIXNEW1SUFFIX' in seqs2 and 'PREFIXNEW2SUFFIX' in seqs2

# Replace with strand='-' reverse complements content before insertion
pp.init()
bg3 = pp.from_seq("ACGT.<region strand='-'>XX</region>.CCCC")
content3 = pp.from_seq('AAA')
result3 = pp.replace_marker_content(bg3, content3, 'region')
df3 = result3.generate_library(num_cycles=1)
seqs3 = list(df3['seq'])
print(f'{seqs3=}')
# AAA reverse complement = TTT
assert df3['seq'].iloc[0] == 'ACGT.TTT.CCCC'

seqs1=['ACGTAAATTTT', 'ACGTGGGTTTT']
seqs2=['PREFIXNEW1SUFFIX', 'PREFIXNEW2SUFFIX']
seqs3=['ACGT.TTT.CCCC']


In [10]:
# --- apply_at_marker: transform content ---
pp.init()

# Apply reverse_complement at marker
bg1 = pp.from_seq('ACGT.<orf>ATGCCC</orf>.TTTT')
result1 = pp.apply_at_marker(bg1, 'orf', pp.reverse_complement)
df1 = result1.generate_library(num_cycles=1)
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
# ATGCCC reverse complement = GGGCAT
assert df1['seq'].iloc[0] == 'ACGT.GGGCAT.TTTT'

# Apply mutagenize at marker (sequential enumeration)
pp.init()
bg2 = pp.from_seq('AAAA.<target>ACGT</target>.TTTT')
result2 = pp.apply_at_marker(
    bg2, 'target',
    lambda p: pp.mutagenize(p, num_mutations=1, mode='sequential')
)
df2 = result2.generate_library(num_cycles=1)
seqs2 = list(df2['seq'])
print(f'{seqs2=}')
# 4 positions x 3 alternatives = 12 single-mutation variants
assert len(df2) == 12
for seq in df2['seq']:
    assert len(seq) == 12+2  # AAAA + 4 + TTTT

# Apply seq_shuffle at marker
pp.init()
bg3 = pp.from_seq('AAA.<region>ACGTACGT</region>.TTT')
result3 = pp.apply_at_marker(bg3, 'region', lambda p: pp.seq_shuffle(p, mode='random'))
df3 = result3.generate_library(num_cycles=1)
seqs3 = list(df3['seq'])
print(f'{seqs3=}')
for seq in df3['seq']:
    assert seq.startswith('AAA') and seq.endswith('TTT')
    assert len(seq[3:-3]) == 8+2  # shuffled 8-char region

seqs1=['ACGT.GGGCAT.TTTT']
seqs2=['AAAA.CCGT.TTTT', 'AAAA.GCGT.TTTT', 'AAAA.TCGT.TTTT', 'AAAA.AAGT.TTTT', 'AAAA.AGGT.TTTT', 'AAAA.ATGT.TTTT', 'AAAA.ACAT.TTTT', 'AAAA.ACCT.TTTT', 'AAAA.ACTT.TTTT', 'AAAA.ACGA.TTTT', 'AAAA.ACGC.TTTT', 'AAAA.ACGG.TTTT']
seqs3=['AAA.GATGCACT.TTT']


In [11]:
# --- remove_marker ---
pp.init()

# Remove marker keeping content (default)
bg1 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
result1 = pp.remove_marker(bg1, 'region', keep_content=True)
df1 = result1.generate_library(num_cycles=1)
seqs1 = list(df1['seq'])
print(f'{seqs1=}')
assert df1['seq'].iloc[0] == 'ACGTTTAAGCGC'


# Remove marker discarding content
pp.init()
bg2 = pp.from_seq('ACGT<region>TTAA</region>GCGC')
result2 = pp.remove_marker(bg2, 'region', keep_content=False)
df2 = result2.generate_library(num_cycles=1)
seqs2 = list(df2['seq'])
print(f'{seqs2=}')
assert df2['seq'].iloc[0] == 'ACGTGCGC'

# Remove zero-length marker
pp.init()
bg3 = pp.from_seq('ACGT<ins/>GCGC')
result3 = pp.remove_marker(bg3, 'ins')
df3 = result3.generate_library(num_cycles=1)
seqs3 = list(df3['seq'])
print(f'{seqs3=}')
assert df3['seq'].iloc[0] == 'ACGTGCGC'

seqs1=['ACGTTTAAGCGC']
seqs2=['ACGTGCGC']
seqs3=['ACGTGCGC']


In [12]:
# --- Marker registration and pool tracking ---
# Auto-registration via from_seq with embedded markers
with pp.Party() as party:
    pool = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    assert party.has_marker('orf')
    m = party.get_marker_by_name('orf')
    assert m.seq_length == 6  # inferred from 'ATGCCC'

# Pool.markers and pool.has_marker
with pp.Party():
    pool = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    assert pool.has_marker('orf')
    assert not pool.has_marker('nonexistent')
    assert len(pool.markers) == 1
    assert any(m.name == 'orf' for m in pool.markers)

# Marker inheritance to child pools
with pp.Party():
    parent = pp.from_seq('AAA<orf>ATGCCC</orf>TTT')
    child = pp.reverse_complement(parent)
    assert child.has_marker('orf')

# Marker removed after replace_marker_content
with pp.Party():
    bg = pp.from_seq('AAA<target>XXXX</target>TTT')
    assert bg.has_marker('target')
    content = pp.from_seq('GGGG')
    result = pp.replace_marker_content(bg, content, 'target')
    assert not result.has_marker('target')

In [13]:
# --- Integration: complete workflows ---
# Workflow 1: insert marker -> transform -> result has no marker tags
with pp.Party():
    bg = pp.from_seq('AAAACGTACGTTTTT')  # 15 chars
    bg_seqs = list(bg.generate_library(num_cycles=1)['seq'])
    print(f'{bg_seqs=}')
    marked = pp.insert_marker(bg, 'target', start=4, stop=12)  # marks CGTACGTT
    marked_seqs = list(marked.generate_library(num_cycles=1)['seq'])
    print(f'{marked_seqs=}')
    result = pp.apply_at_marker(marked, 'target', pp.reverse_complement)
    result_seqs = list(result.generate_library(num_cycles=1)['seq'])
    print(f'{result_seqs=}')
    assert '<' not in result_seqs[0]  # no marker tags in output
    assert len(result_seqs[0]) == 15

# Workflow 2: marker_scan + replace_marker_content
with pp.Party():
    marked = pp.marker_scan('ACGTACGT', marker='ins', positions=[2, 6], mode='sequential')
    inserts = pp.from_seq('NNN')
    result = pp.replace_marker_content(marked, inserts, 'ins')
    df = result.generate_library(num_cycles=1)
    for seq in df['seq']:
        assert 'NNN' in seq
        assert '<' not in seq  # no markers left

# Workflow 3: mutagenize_orf preserves markers
with pp.Party():
    orf_with_marker = 'ATGTGT<site/>GGTTAA'  # ATG TGT GGT TAA
    pool = pp.from_seq(orf_with_marker)
    mutated = pp.mutagenize_orf(pool, num_mutations=1, codon_positions=[1], mode='random')
    df = mutated.generate_library(num_seqs=5, seed=42)
    for seq in df['seq']:
        assert '<site/>' in seq  # marker preserved
        clean = strip_all_markers(seq)
        assert clean[:3] == 'ATG' and clean[-3:] == 'TAA'  # start/stop intact

bg_seqs=['AAAACGTACGTTTTT']
marked_seqs=['AAAA<target>CGTACGTT</target>TTT']
result_seqs=['AAAAAACGTACGTTT']


In [15]:
pp.init()
result = pp.marker_scan('ACGTACGT', marker='ins', mode='sequential', strand='both')
df = result.generate_library(num_cycles=1)
display(df['seq'])


0     <ins strand='+'/>ACGTACGT
1     <ins strand='-'/>ACGTACGT
2     A<ins strand='+'/>CGTACGT
3     A<ins strand='-'/>CGTACGT
4     AC<ins strand='+'/>GTACGT
5     AC<ins strand='-'/>GTACGT
6     ACG<ins strand='+'/>TACGT
7     ACG<ins strand='-'/>TACGT
8     ACGT<ins strand='+'/>ACGT
9     ACGT<ins strand='-'/>ACGT
10    ACGTA<ins strand='+'/>CGT
11    ACGTA<ins strand='-'/>CGT
12    ACGTAC<ins strand='+'/>GT
13    ACGTAC<ins strand='-'/>GT
14    ACGTACG<ins strand='+'/>T
15    ACGTACG<ins strand='-'/>T
16    ACGTACGT<ins strand='+'/>
17    ACGTACGT<ins strand='-'/>
Name: seq, dtype: object